<a href="https://colab.research.google.com/github/YB441/codsoft/blob/main/spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import string
import pickle
import warnings
import textwrap
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
import matplotlib
matplotlib.use("Agg")                       # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB # Added ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay # Added for Precision-Recall Curve
)

# Download NLTK data if not already present
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True) # Open Multilingual Wordnet for lemmatizer

warnings.filterwarnings("ignore")
# ─── Configuration ────────────────────────────────────────────────────────────
DATASET_PATH  = "spam.csv"
OUTPUT_DIR    = "outputs"
MODEL_DIR     = "saved_models"
RANDOM_STATE  = 42
TEST_SIZE     = 0.20
# Colour palette
PALETTE = {
    "ham":       "#2ecc71",   # green
    "spam":      "#e74c3c",   # red
    "accent":    "#3498db",   # blue
    "bg_dark":   "#1a1a2e",
    "bg_card":   "#16213e",
    "text":      "#eaeaea",
}
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.facecolor": PALETTE["bg_dark"],
    "axes.facecolor":   PALETTE["bg_card"],
    "axes.edgecolor":   PALETTE["text"],
    "axes.labelcolor":  PALETTE["text"],
    "text.color":       PALETTE["text"],
    "xtick.color":      PALETTE["text"],
    "ytick.color":      PALETTE["text"],
    "figure.dpi":       150,
    "savefig.bbox":     "tight",
    "savefig.facecolor": PALETTE["bg_dark"],
})
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
# ═════════════════════════════════════════════════════════════════════════════
# STEP 1 — DATA LOADING & CLEANING
# ═════════════════════════════════════════════════════════════════════════════
def load_and_clean(path: str) -> pd.DataFrame:
    """Load the SMS Spam Collection CSV and return a clean DataFrame."""
    print("=" * 70)
    print("  STEP 1 · DATA LOADING & CLEANING")
    print("=" * 70)
    df = pd.read_csv(path, encoding="latin-1")
    # The CSV has extra unnamed columns — drop them
    df = df[["v1", "v2"]].copy()
    df.columns = ["label", "message"]
    # Binary encode
    df["label_num"] = df["label"].map({"ham": 0, "spam": 1})
    # Derived features for EDA
    df["msg_length"]  = df["message"].apply(len)
    df["word_count"]  = df["message"].apply(lambda x: len(x.split()))
    df["digit_count"] = df["message"].apply(lambda x: sum(c.isdigit() for c in x))
    df["upper_count"] = df["message"].apply(lambda x: sum(c.isupper() for c in x))
    df["special_count"] = df["message"].apply(
        lambda x: sum(c in string.punctuation for c in x)
    )
    print(f"  ✓ Loaded {len(df):,} messages")
    print(f"  ✓ Ham  : {(df['label'] == 'ham').sum():,}")
    print(f"  ✓ Spam : {(df['label'] == 'spam').sum():,}")
    print(f"  ✓ Null values: {df.isnull().sum().sum()}")
    print(f"  ✓ Duplicates : {df.duplicated(subset='message').sum()}")
    print()
    return df
# ═════════════════════════════════════════════════════════════════════════════
# STEP 2 — EXPLORATORY DATA ANALYSIS
# ═════════════════════════════════════════════════════════════════════════════
def plot_class_distribution(df: pd.DataFrame):
    """Pie + bar chart showing class balance."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    counts = df["label"].value_counts()
    colours = [PALETTE["ham"], PALETTE["spam"]]
    # — Pie chart —
    wedges, texts, autotexts = axes[0].pie(
        counts, labels=["Ham", "Spam"], autopct="%1.1f%%",
        colors=colours, startangle=140,
        textprops={"color": PALETTE["text"], "fontsize": 13, "fontweight": "bold"},
        wedgeprops={"edgecolor": PALETTE["bg_dark"], "linewidth": 2},
        explode=(0, 0.06),
    )
    axes[0].set_title("Class Distribution (Pie)", fontsize=14, fontweight="bold", pad=15)
    # — Bar chart —
    bars = axes[1].bar(["Ham", "Spam"], counts.values, color=colours,
                       edgecolor=PALETTE["bg_dark"], linewidth=1.5, width=0.5)
    for bar, val in zip(bars, counts.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
                     f"{val:,}", ha="center", va="bottom",
                     fontsize=13, fontweight="bold", color=PALETTE["text"])
    axes[1].set_title("Class Distribution (Bar)", fontsize=14, fontweight="bold", pad=15)
    axes[1].set_ylabel("Count")
    axes[1].set_ylim(0, counts.max() * 1.15)
    fig.suptitle("📊 Dataset Class Balance", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "01_class_distribution.png"))
    plt.close(fig)
    print("  ✓ Saved 01_class_distribution.png")
def plot_message_length_distribution(df: pd.DataFrame):
    """Histogram + box plot of message lengths by class."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colours = [PALETTE["ham"], PALETTE["spam"]]
    for label, colour in zip(["ham", "spam"], colours):
        subset = df[df["label"] == label]["msg_length"]
        axes[0].hist(subset, bins=60, alpha=0.65, color=colour, label=label.capitalize(),
                     edgecolor=PALETTE["bg_dark"])
    axes[0].set_title("Message Length Distribution", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("Character Length")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()
    data_to_plot = [df[df["label"] == "ham"]["msg_length"],
                    df[df["label"] == "spam"]["msg_length"]]
    bp = axes[1].boxplot(data_to_plot, labels=["Ham", "Spam"], patch_artist=True,
                         boxprops=dict(linewidth=1.5),
                         medianprops=dict(color="white", linewidth=2))
    for patch, colour in zip(bp["boxes"], colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.75)
    axes[1].set_title("Message Length Box Plot", fontsize=14, fontweight="bold")
    axes[1].set_ylabel("Character Length")
    fig.suptitle("📏 Message Length Analysis", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "02_message_length.png"))
    plt.close(fig)
    print("  ✓ Saved 02_message_length.png")
def plot_feature_comparison(df: pd.DataFrame):
    """Compare engineered features (digit count, uppercase, special chars) by class."""
    features = ["word_count", "digit_count", "upper_count", "special_count"]
    titles   = ["Word Count", "Digit Count", "Uppercase Letters", "Special Characters"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    for ax, feat, title in zip(axes, features, titles):
        ham_vals  = df[df["label"] == "ham"][feat]
        spam_vals = df[df["label"] == "spam"][feat]
        ax.hist(ham_vals, bins=40, alpha=0.6, color=PALETTE["ham"], label="Ham",
                edgecolor=PALETTE["bg_dark"])
        ax.hist(spam_vals, bins=40, alpha=0.6, color=PALETTE["spam"], label="Spam",
                edgecolor=PALETTE["bg_dark"])
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_ylabel("Frequency")
        ax.legend(fontsize=10)
    fig.suptitle("🔍 Feature Comparison: Ham vs Spam", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "03_feature_comparison.png"))
    plt.close(fig)
    print("  ✓ Saved 03_feature_comparison.png")
def plot_correlation_heatmap(df: pd.DataFrame):
    """Heatmap of numeric feature correlations."""
    numeric_cols = ["msg_length", "word_count", "digit_count",
                    "upper_count", "special_count", "label_num"]
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    cmap = LinearSegmentedColormap.from_list("custom", ["#3498db", "#1a1a2e", "#e74c3c"])
    sns.heatmap(corr, annot=True, fmt=".2f", cmap=cmap, linewidths=0.5,
                linecolor=PALETTE["bg_dark"], ax=ax, vmin=-1, vmax=1,
                annot_kws={"fontsize": 11, "fontweight": "bold"})
    ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold", pad=15)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "04_correlation_heatmap.png"))
    plt.close(fig)
    print("  ✓ Saved 04_correlation_heatmap.png")
def plot_wordclouds(df: pd.DataFrame):
    """Word clouds for ham and spam messages."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, label, title, colour in [
        (axes[0], "ham",  "☁️ Ham Word Cloud",  PALETTE["ham"]),
        (axes[1], "spam", "☁️ Spam Word Cloud", PALETTE["spam"]),
    ]:
        text = " ".join(df[df["label"] == label]["message"].tolist())
        wc = WordCloud(
            width=800, height=400,
            background_color=PALETTE["bg_dark"],
            colormap="Greens" if label == "ham" else "Reds",
            max_words=150,
            contour_width=1,
            contour_color=colour,
        ).generate(text)
        ax.imshow(wc, interpolation="bilinear")
        ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "05_wordclouds.png"))
    plt.close(fig)
    print("  ✓ Saved 05_wordclouds.png")
def run_eda(df: pd.DataFrame):
    """Execute all EDA visualizations."""
    print("=" * 70)
    print("  STEP 2 · EXPLORATORY DATA ANALYSIS")
    print("=" * 70)
    plot_class_distribution(df)
    plot_message_length_distribution(df)
    plot_feature_comparison(df)
    plot_correlation_heatmap(df)
    plot_wordclouds(df)
    # Print numeric insights
    print()
    print("  ┌──────────────────────────────────────────────────────┐")
    print("  │              📈  KEY STATISTICAL INSIGHTS            │")
    print("  ├──────────────────────────────────────────────────────┤")
    for label in ["ham", "spam"]:
        sub = df[df["label"] == label]
        print(f"  │  {label.upper():5s} Messages                                  │")
        print(f"  │    Avg length   : {sub['msg_length'].mean():>7.1f} chars                 │")
        print(f"  │    Avg words    : {sub['word_count'].mean():>7.1f}                       │")
        print(f"  │    Avg digits   : {sub['digit_count'].mean():>7.1f}                       │")
        print(f"  │    Avg uppercase: {sub['upper_count'].mean():>7.1f}                       │")
        print(f"  │    Avg specials : {sub['special_count'].mean():>7.1f}                       │")
        print(f"  ├──────────────────────────────────────────────────┤")
    print(f"  │  ⚠  Spam messages are on avg "
          f"{df[df['label']=='spam']['msg_length'].mean() / df[df['label']=='ham']['msg_length'].mean():.1f}x longer      │")
    print(f"  │  ⚠  Spam has "
          f"{df[df['label']=='spam']['digit_count'].mean() / max(df[df['label']=='ham']['digit_count'].mean(), 1):.1f}x more digits (phone nos, codes)   │")
    print("  └──────────────────────────────────────────────────────┘")
    print()
# ═════════════════════════════════════════════════════════════════════════════
# STEP 3 — TEXT PREPROCESSING
# ═════════════════════════════════════════════════════════════════════════════
# Initialize lemmatizer outside the function for efficiency
lemmatizer = WordNetLemmatizer()

# Define custom stopwords
# Keep words that might indicate urgency/spamminess
custom_stopwords = set(stopwords.words('english'))
words_to_keep = {'now', 'free', 'won', 'urgent', 'claim', 'guarantee', 'prize', 'cash', 'winner', 'offer'}
custom_stopwords = [word for word in custom_stopwords if word not in words_to_keep]

def preprocess_text(text: str) -> str:
    """Clean a single SMS message for modelling."""
    text = text.lower()                                        # lowercase
    # Replace URLs, phone numbers, and emails with generic tokens
    text = re.sub(r'http\S+|www\S+|https\S+', 'URL_TOKEN', text) # URLs
    text = re.sub(r'\b\d{10,}\b', 'PHONENUM_TOKEN', text)    # Phone numbers (10+ digits)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'EMAIL_TOKEN', text) # Email addresses

    text = re.sub(r'<.*?>', ' ', text)                         # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)                  # remove punctuation, emojis, special characters
    text = re.sub(r'\s+', ' ', text).strip()                  # collapse whitespace

    # Lemmatization and stopword removal
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words if word not in custom_stopwords]
    return ' '.join(lemmatized_words)

def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Apply text preprocessing to the entire DataFrame."""
    print("=" * 70)
    print("  STEP 3 · TEXT PREPROCESSING")
    print("=" * 70)
    df["clean_message"] = df["message"].apply(preprocess_text)
    print("  ✓ Lowercased all text")
    print("  ✓ Replaced URLs, phone numbers, emails with tokens")
    print("  ✓ Removed HTML tags, punctuation, emojis, special characters")
    print("  ✓ Applied Lemmatization instead of stemming")
    print("  ✓ Filtered stopwords (keeping urgency words)")
    print("  ✓ Collapsed whitespace")
    print()
    print("  Sample transformations:")
    for i in range(3):
        orig  = textwrap.shorten(df.iloc[i]["message"], width=60, placeholder="…")
        clean = textwrap.shorten(df.iloc[i]["clean_message"], width=60, placeholder="…")
        print(f"    BEFORE: {orig}")
        print(f"    AFTER : {clean}")
        print()
    return df
# ═════════════════════════════════════════════════════════════════════════════
# STEP 4 — TF-IDF FEATURE EXTRACTION
# ═════════════════════════════════════════════════════════════════════════════
def extract_features(X_train, X_test):
    """Fit TF-IDF vectorizer and transform train/test sets."""
    print("=" * 70)
    print("  STEP 4 · TF-IDF FEATURE EXTRACTION")
    print("=" * 70)
    vectorizer = TfidfVectorizer(
        stop_words=custom_stopwords, # Use custom stopwords
        max_features=5000,
        ngram_range=(1, 2),      # unigrams + bigrams
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,       # apply 1 + log(tf)
    )
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    print(f"  ✓ Vocabulary size     : {len(vectorizer.vocabulary_):,} terms")
    print(f"  ✓ N-gram range        : (1, 2)  — unigrams + bigrams")
    print(f"  ✓ Sublinear TF        : enabled (1 + log(tf)) ")
    print(f"  ✓ Train matrix shape  : {X_train_tfidf.shape}")
    print(f"  ✓ Test  matrix shape  : {X_test_tfidf.shape}")
    print()
    return vectorizer, X_train_tfidf, X_test_tfidf
# ═════════════════════════════════════════════════════════════════════════════
# STEP 5 & 6 — MODEL TRAINING, CROSS-VALIDATION & EVALUATION
# ═════════════════════════════════════════════════════════════════════════════
def train_and_evaluate(X_train, X_test, y_train, y_test, vectorizer):
    """Train models, cross-validate, and evaluate with full metrics."""
    print("=" * 70)
    print("  STEP 5 · MODEL TRAINING & CROSS-VALIDATION")
    print("=" * 70)

    # Define classifiers with class_weight and ComplementNB
    classifiers = {
        "Naive Bayes": MultinomialNB(),
        "Complement Naive Bayes": ComplementNB(), # Added ComplementNB
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'), # Added class_weight
        "Support Vector Machine": LinearSVC(max_iter=2000, class_weight='balanced', dual=False) # Added class_weight
    }

    # Define parameter grids for GridSearchCV
    param_grids = {
        "Naive Bayes": {
            'alpha': [0.1, 0.5, 1.0, 2.0]
        },
        "Complement Naive Bayes": {
            'alpha': [0.1, 0.5, 1.0, 2.0]
        },
        "Logistic Regression": {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'] # liblinear supports l1
        },
        "Support Vector Machine": {
            'C': [0.1, 1.0, 10.0],
            'loss': ['hinge', 'squared_hinge']
        }
    }

    print("\n--- Training and Evaluating Models ---")
    results = {}
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for name, clf in classifiers.items():
        print(f"\n  ╔══════════════════════════════════════════════╗")
        print(f"  ║  🤖  {name:<40s} ║")
        print(f"  ╚══════════════════════════════════════════════╝")

        # Perform GridSearchCV for hyperparameter tuning
        print(f"    Tuning hyperparameters for {name}...")
        grid_search = GridSearchCV(clf, param_grids[name], cv=cv_strategy,
                                   scoring='f1', n_jobs=-1, verbose=0)
        grid_search.fit(X_train, y_train) # Note: X_train here is the TF-IDF matrix

        best_clf = grid_search.best_estimator_
        best_params = grid_search.best_params_
        print(f"    Best parameters     : {best_params}")
        print(f"    Best CV F1-Score    : {grid_search.best_score_:.4f}")

        # Evaluate the best model on the test set
        y_pred = best_clf.predict(X_test)

        # Decision function / probability for ROC and Precision-Recall
        if hasattr(best_clf, "predict_proba"):
            y_score = best_clf.predict_proba(X_test)[:, 1]
        elif hasattr(best_clf, "decision_function"):
            y_score = best_clf.decision_function(X_test)
        else:
            y_score = y_pred.astype(float)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec  = recall_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)
        auc  = roc_auc_score(y_test, y_score)

        results[name] = {
            "model": best_clf, "y_pred": y_pred, "y_score": y_score,
            "accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "auc": auc, "cv_f1_mean": grid_search.best_score_,
            "best_params": best_params
        }

        print(f"\n    ┌──────────────┬───────────┐")
        print(f"    │ Metric       │   Score   │")
        print(f"    ├──────────────┼───────────┤")
        print(f"    │ Accuracy     │  {acc:.4f}   │")
        print(f"    │ Precision    │  {prec:.4f}   │")
        print(f"    │ Recall       │  {rec:.4f}   │")
        print(f"    │ F1-Score     │  {f1:.4f}   │")
        print(f"    │ ROC-AUC      │  {auc:.4f}   │")
        print(f"    └──────────────┴───────────┘")
        print(f"\n    Classification Report:")
        report = classification_report(y_test, y_pred, target_names=["Ham", "Spam"], digits=4)
        for line in report.split("\n"):
            print(f"      {line}")

    return results
# ═════════════════════════════════════════════════════════════════════════════
# STEP 7 — VISUALIZATIONS (Confusion Matrices, ROC, Comparison, Precision-Recall)
# ═════════════════════════════════════════════════════════════════════════════
def plot_confusion_matrices(results, y_test):
    """Side-by-side confusion matrices for all models."""
    # Adjust subplot grid based on number of models
    num_models = len(results)
    if num_models <= 3:
        fig, axes = plt.subplots(1, num_models, figsize=(6 * num_models, 5))
        if num_models == 1: axes = [axes] # Ensure axes is iterable
    else:
        rows = (num_models + 1) // 2
        fig, axes = plt.subplots(rows, 2, figsize=(12, 5 * rows))
        axes = axes.flatten()

    colours_cm = LinearSegmentedColormap.from_list("cm", [PALETTE["bg_card"], PALETTE["spam"]])

    for i, (name, res) in enumerate(results.items()):
        cm = confusion_matrix(y_test, res["y_pred"])
        disp = ConfusionMatrixDisplay(cm, display_labels=["Ham", "Spam"])
        disp.plot(ax=axes[i], cmap=colours_cm, colorbar=False,
                  text_kw={"fontsize": 16, "fontweight": "bold", "color": PALETTE["text"]})
        axes[i].set_title(name, fontsize=13, fontweight="bold", pad=10)
        axes[i].tick_params(axis='x', colors=PALETTE["text"])
        axes[i].tick_params(axis='y', colors=PALETTE["text"])
        axes[i].xaxis.label.set_color(PALETTE["text"])
        axes[i].yaxis.label.set_color(PALETTE["text"])

    fig.suptitle("🔢 Confusion Matrices", fontsize=16, fontweight="bold", y=1.03)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "06_confusion_matrices.png"))
    plt.close(fig)
    print("  ✓ Saved 06_confusion_matrices.png")

def plot_precision_recall_curves(results, y_test):
    """Overlay Precision-Recall curves for all classifiers."""
    fig, ax = plt.subplots(figsize=(8, 6))
    model_colours = [PALETTE["ham"], PALETTE["accent"], PALETTE["spam"], "#f1c40f"]

    for i, (name, res) in enumerate(results.items()):
        PrecisionRecallDisplay.from_predictions(y_test, res["y_score"], ax=ax,
                                                name=name, color=model_colours[i % len(model_colours)])

    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title("📈 Precision-Recall Curves — Model Comparison", fontsize=14, fontweight="bold", pad=15)
    ax.legend(loc="lower left", fontsize=11)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "10_precision_recall_curves.png")) # New filename
    plt.close(fig)
    print("  ✓ Saved 10_precision_recall_curves.png")

def plot_roc_curves(results, y_test):
    """Overlay ROC curves for all classifiers."""
    fig, ax = plt.subplots(figsize=(8, 6))
    model_colours = [PALETTE["ham"], PALETTE["accent"], PALETTE["spam"], "#f1c40f"]
    for i, (name, res) in enumerate(results.items()):
        fpr, tpr, _ = roc_curve(y_test, res["y_score"])
        ax.plot(fpr, tpr, color=model_colours[i % len(model_colours)], linewidth=2.5,
                label=f"{name} (AUC = {res['auc']:.4f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, alpha=0.6, label="Random")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("📈 ROC Curves — Model Comparison", fontsize=14, fontweight="bold", pad=15)
    ax.legend(loc="lower right", fontsize=11)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "07_roc_curves.png"))
    plt.close(fig)
    print("  ✓ Saved 07_roc_curves.png")
def plot_model_comparison(results):
    """Grouped bar chart comparing all metrics across models."""
    metrics = ["accuracy", "precision", "recall", "f1", "auc"]
    labels  = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
    model_names = list(results.keys())
    model_colours = [PALETTE["ham"], PALETTE["accent"], PALETTE["spam"], "#f1c40f"]
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names) # Adjust width dynamically
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (name) in enumerate(model_names):
        values = [results[name][m] for m in metrics]
        bars = ax.bar(x + i * width, values, width, label=name, color=model_colours[i % len(model_colours)],
                      edgecolor=PALETTE["bg_dark"], linewidth=1)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9,
                    fontweight="bold", color=PALETTE["text"])
    ax.set_xticks(x + (len(model_names) - 1) * width / 2)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0.5, 1.05)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("🏆 Model Performance Comparison", fontsize=14, fontweight="bold", pad=15)
    ax.legend(fontsize=11)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "08_model_comparison.png"))
    plt.close(fig)
    print("  ✓ Saved 08_model_comparison.png")

def plot_top_features(vectorizer, results):
    """Show top TF-IDF features (most predictive words) for each class."""
    # Use Logistic Regression (or best linear model) coefficients for interpretability
    # Prioritize Logistic Regression if available, otherwise LinearSVC
    if "Logistic Regression" in results:
        model_name = "Logistic Regression"
    elif "Support Vector Machine" in results:
        model_name = "Support Vector Machine"
    else:
        print("  ✗ No linear model found for feature importance visualization.")
        return

    lr = results[model_name]["model"]

    # Ensure the model has coefficients (e.g., if it's a linear model)
    if not hasattr(lr, 'coef_'):
        print(f"  ✗ Model {model_name} does not have 'coef_' attribute for feature importance.")
        return

    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = lr.coef_[0]
    top_n = 15
    top_spam_idx = np.argsort(coefs)[-top_n:][::-1]
    top_ham_idx  = np.argsort(coefs)[:top_n]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    # Top spam words
    axes[0].barh(range(top_n), coefs[top_spam_idx], color=PALETTE["spam"],
                 edgecolor=PALETTE["bg_dark"])
    axes[0].set_yticks(range(top_n))
    axes[0].set_yticklabels(feature_names[top_spam_idx], fontsize=11)
    axes[0].invert_yaxis()
    axes[0].set_title("🚨 Top Spam Indicators", fontsize=13, fontweight="bold", pad=12)
    axes[0].set_xlabel("TF-IDF Coefficient Weight")
    # Top ham words
    axes[1].barh(range(top_n), np.abs(coefs[top_ham_idx]), color=PALETTE["ham"],
                 edgecolor=PALETTE["bg_dark"])
    axes[1].set_yticks(range(top_n))
    axes[1].set_yticklabels(feature_names[top_ham_idx], fontsize=11)
    axes[1].invert_yaxis()
    axes[1].set_title("✅ Top Ham Indicators", fontsize=13, fontweight="bold", pad=12)
    axes[1].set_xlabel("TF-IDF Coefficient Weight (abs)")
    fig.suptitle("🔑 Feature Importance — Most Predictive Words",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "09_feature_importance.png"))
    plt.close(fig)
    print("  ✓ Saved 09_feature_importance.png")
# ═════════════════════════════════════════════════════════════════════════════
# STEP 8 — LIVE PREDICTION DEMO
# ═════════════════════════════════════════════════════════════════════════════
def live_demo(best_model, vectorizer, best_name):
    """Predict on unseen example messages."""
    print("\n" + "=" * 70)
    print("  STEP 8 · LIVE PREDICTION DEMO")
    print("=" * 70)
    test_messages = [
        "Hey, are we still meeting for lunch tomorrow?",
        "Congratulations! You've won a £1000 cash prize! Call 0800-123-456 NOW!",
        "Reminder: Your dentist appointment is at 3 PM today.",
        "URGENT! Your mobile number has been selected for a £2000 prize. Claim now!",
        "Can you pick up some milk on your way home?",
        "FREE entry to win a brand new iPhone! Text WIN to 80808 right now!",
        "I'll be there in 10 minutes, stuck in traffic.",
        "You have been chosen to receive a special offer. Reply STOP to opt out.",
    ]
    print(f"\n  Using best model: {best_name}\n")
    print("  ┌──────────────────────────────────────────────────────────────────┐")
    for msg in test_messages:
        clean = preprocess_text(msg)
        vec   = vectorizer.transform([clean])
        pred  = best_model.predict(vec)[0]
        label = "🚨 SPAM" if pred == 1 else "✅ HAM "
        short_msg = textwrap.shorten(msg, width=52, placeholder="…")
        print(f"  │  {label}  │ {short_msg:<52s} │")
    print("  └──────────────────────────────────────────────────────────────────┘")
# ═════════════════════════════════════════════════════════════════════════════
# STEP 9 — SAVE BEST MODEL
# ═════════════════════════════════════════════════════════════════════════════
def save_best_model(results, vectorizer):
    """Pickle the best performing model and its vectorizer."""
    print("\n" + "=" * 70)
    print("  STEP 9 · SAVING BEST MODEL")
    print("=" * 70)
    # Determine best model based on F1-score
    best_name  = max(results, key=lambda k: results[k]["f1"])
    best_model = results[best_name]["model"]
    model_path = os.path.join(MODEL_DIR, "best_model.pkl")
    vec_path   = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    with open(vec_path, "wb") as f:
        pickle.dump(vectorizer, f)
    print(f"  ✓ Best model        : {best_name}")
    print(f"  ✓ Best F1-Score     : {results[best_name]['f1']:.4f}")
    print(f"  ✓ Model saved to    : {model_path}")
    print(f"  ✓ Vectorizer saved  : {vec_path}")
    print()
    return best_name, best_model
# ═════════════════════════════════════════════════════════════════════════════
# STEP 10 — FINAL SUMMARY REPORT
# ═════════════════════════════════════════════════════════════════════════════
def print_final_summary(results, best_name):
    """Print a formatted summary table."""
    print("=" * 70)
    print("  ★  FINAL SUMMARY REPORT")
    print("=" * 70)
    print()
    print("  ┌─────────────────────────┬──────────┬───────────┬────────┬────────┬────────┐")
    print("  │ Model                   │ Accuracy │ Precision │ Recall │   F1   │  AUC   │")
    print("  ├─────────────────────────┼──────────┼───────────┼────────┼────────┼────────┤")
    for name, res in results.items():
        marker = " 🏆" if name == best_name else "   "
        print(f"  │ {name:<23s} │  {res['accuracy']:.4f}  │  {res['precision']:.4f}   │ {res['recall']:.4f} │ {res['f1']:.4f} │ {res['auc']:.4f} │{marker}")
    print("  └─────────────────────────┴──────────┴───────────┴────────┴────────┴────────┘")
    print(f"\n  🏆 Winner: {best_name}")
    print(f"  Best parameters for {best_name}: {results[best_name]['best_params']}")
    print()
    print("  📁 All outputs saved to:")
    print(f"     • Visualizations : ./{OUTPUT_DIR}/")
    print(f"     • Saved model    : ./{MODEL_DIR}/")
    print()
    print("=" * 70)
    print("  Pipeline complete. Happy classifying! 🎉")
    print("=" * 70)
# ═════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def main():
    print()
    print("╔══════════════════════════════════════════════════════════════════════╗")
    print("║        SMS SPAM DETECTION — AI CLASSIFICATION PIPELINE             ║")
    print("║        Naive Bayes · Logistic Regression · SVM · ComplementNB      ║")
    print("╚══════════════════════════════════════════════════════════════════════╝")
    print()
    # 1. Load & clean
    df = load_and_clean(DATASET_PATH)
    # 2. EDA
    run_eda(df)
    # 3. Preprocess
    df = preprocess_dataset(df)
    # 4. Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_message"], df["label_num"],
        test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label_num"]
    )
    print(f"  Train set: {len(X_train):,} | Test set: {len(X_test):,}")
    print()
    # 5. TF-IDF
    vectorizer, X_train_tfidf, X_test_tfidf = extract_features(X_train, X_test)
    # 6. Train & evaluate
    results = train_and_evaluate(X_train_tfidf, X_test_tfidf, y_train, y_test, vectorizer)
    # 7. Visualizations
    print("\n" + "=" * 70)
    print("  STEP 7 · GENERATING VISUALIZATIONS")
    print("=" * 70)
    plot_confusion_matrices(results, y_test)
    plot_roc_curves(results, y_test)
    plot_precision_recall_curves(results, y_test) # Added Precision-Recall Curve
    plot_model_comparison(results)
    plot_top_features(vectorizer, results)
    print()
    # 8. Save best model
    best_name, best_model = save_best_model(results, vectorizer)
    # 9. Live demo
    live_demo(best_model, vectorizer, best_name)
    # 10. Final summary
    print()
    print_final_summary(results, best_name)
if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════════════════╗
║        SMS SPAM DETECTION — AI CLASSIFICATION PIPELINE             ║
║        Naive Bayes · Logistic Regression · SVM · ComplementNB      ║
╚══════════════════════════════════════════════════════════════════════╝

  STEP 1 · DATA LOADING & CLEANING
  ✓ Loaded 5,572 messages
  ✓ Ham  : 4,825
  ✓ Spam : 747
  ✓ Null values: 0
  ✓ Duplicates : 403

  STEP 2 · EXPLORATORY DATA ANALYSIS
  ✓ Saved 01_class_distribution.png
  ✓ Saved 02_message_length.png
  ✓ Saved 03_feature_comparison.png
  ✓ Saved 04_correlation_heatmap.png
  ✓ Saved 05_wordclouds.png

  ┌──────────────────────────────────────────────────────┐
  │              📈  KEY STATISTICAL INSIGHTS            │
  ├──────────────────────────────────────────────────────┤
  │  HAM   Messages                                  │
  │    Avg length   :    71.0 chars                 │
  │    Avg words    :    14.2                       │
  │    Avg digits   :     0.3 